<a href="https://colab.research.google.com/github/AntoninLeSecq/ETL-AirLife/blob/main/Donn%C3%A9es%20communes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd


def extract_data(list_chemin):

    # print("📄 Reading tourism data from CSV...")

    try:

        df = pd.DataFrame()
        for chemin in list_chemin:
            df = pd.concat([df, pd.read_csv("../data/"+ chemin)], ignore_index=True)

        print('Data retreived in a dataframe')
        return df

    except Exception as e:
        print(f"❌ Error reading data: {e}")
        return pd.DataFrame()

In [13]:
df = pd.DataFrame(pd.read_csv("communes-france-2025.csv"))
df.shape

/tmp/ipython-input-1236332959.py:1: DtypeWarning: Columns (1,12,14,16,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.DataFrame(pd.read_csv("communes-france-2025.csv"))


(34935, 47)

In [19]:
#On ne garde que le 'nom_standard' dans pour la dénomination
df.drop(['nom_sans_pronom', 'nom_a', 'nom_de', 'nom_sans_accent','nom_standard_majuscule'], axis=1, inplace=True)
df.drop(['typecom', 'typecom_texte', 'canton_code', 'canton_nom', 'reg_code', 'dep_code', 'epci_code', 'epci_nom', 'codes_postaux', 'academie_code',
         'academie_nom', 'code_unite_urbaine', 'nom_unite_urbaine', 'taille_unite_urbaine',
         'type_commune_unite_urbaine', 'statut_commune_unite_urbaine', 'superficie_hectare',
         'altitude_moyenne', 'altitude_minimale','altitude_maximale', 'latitude_centre', 'longitude_centre',
         'niveau_equipements_services', 'niveau_equipements_services_texte', 'gentile', 'url_wikipedia', 'url_villedereve'], axis=1, inplace=True)


In [5]:
df.shape
df.tail()

,Unnamed: 0,code_insee,nom_standard,reg_nom,dep_nom,code_postal,zone_emploi,code_insee_centre_zone_emploi,population,superficie_km2,densite,latitude_mairie,longitude_mairie,grille_densite,grille_densite_texte
34930,34930,97613,M'Tsangamouji,Mayotte,Mayotte,97650.0,601.0,NaN,6432,22,298.0,-12.761,45.084,3,Petites villes
34931,34931,97614,Ouangani,Mayotte,Mayotte,97670.0,601.0,NaN,10203,18,558.0,-12.849,45.139,3,Petites villes
34932,34932,97615,Pamandzi,Mayotte,Mayotte,97615.0,601.0,NaN,11442,4,2686.0,-12.798,45.275,2,Centres urbains intermédiaires
34933,34933,97616,Sada,Mayotte,Mayotte,97640.0,601.0,NaN,11156,11,1028.0,-12.847,45.106,2,Centres urbains intermédiaires
34934,34934,97617,Tsingoni,Mayotte,Mayotte,97680.0,601.0,NaN,13934,34,407.0,-12.790,45.105,2,Centres urbains intermédiaires


* code_insee: Code commune, Code INSEE, Code assigné par l'INSEE à la commune
* nom_standard: Nom normalisé de la commune, avec son article (ex: Le Havre)
* reg_nom: Nom de la région où est située la commune
* dep_nom: Nom du département où est située la commune
* code_postal: Code postal principal la commune
* zone_emploi: Zone d'emploi de la commune, défini par l'INSEE
* code_insee_centre_zone_emploi: Code INSEE de la commune centre de la zone d'emploi
* population: Population municipale
* superficie_km2: Superficie de la commune, en km2
* densite: Densité de la commune, en habitant au km2
* latitude_mairie: Latitude de la mairie
* longitude_mairie: Longitude de la mairie
* grille_densite: Grille communale de densité à 7 niveaux, selon l'INSEE
* grille_densite_texte: Texte de la grille communale de densité à 7 niveaux, selon l'INSEE



In [29]:
#On enlève les départements d'Outre mer
df =df[~df['reg_nom'].isin(['Mayotte', 'Guyane', 'La Réunion', 'Guadeloupe', 'Martinique'])]
df.tail()

# il y a 3 petites communes sans code postal ni zone emploi dans la db qu'on supprime pour simplifier
df = df.dropna(subset=['code_insee_centre_zone_emploi', 'code_postal'])

df['code_cluster'] = pd.factorize(df['code_insee_centre_zone_emploi'])[0] + 1

df

,Unnamed: 0,code_insee,nom_standard,reg_nom,dep_nom,code_postal,zone_emploi,code_insee_centre_zone_emploi,population,superficie_km2,densite,latitude_mairie,longitude_mairie,grille_densite,grille_densite_texte,code_cluster
0,0,01001,L'Abergement-Clémenciat,Auvergne-Rhône-Alpes,Ain,1400.0,8405.0,01053,832,16,53.0,46.151,4.921,6,Rural à habitat dispersé,1
1,1,01002,L'Abergement-de-Varey,Auvergne-Rhône-Alpes,Ain,1640.0,8405.0,01053,267,9,29.0,46.007,5.423,6,Rural à habitat dispersé,1
2,2,01004,Ambérieu-en-Bugey,Auvergne-Rhône-Alpes,Ain,1500.0,8405.0,01053,14854,24,607.0,45.958,5.360,2,Centres urbains intermédiaires,1
3,3,01005,Ambérieux-en-Dombes,Auvergne-Rhône-Alpes,Ain,1330.0,8434.0,69264,1897,16,118.0,45.996,4.903,5,Bourgs ruraux,2
4,4,01006,Ambléon,Auvergne-Rhône-Alpes,Ain,1300.0,8404.0,01034,113,6,19.0,45.748,5.601,6,Rural à habitat dispersé,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34801,34801,95676,Villers-en-Arthies,Île-de-France,Val-d'Oise,95510.0,1101.0,95127.0,489,8,58.0,49.088,1.725,6,Rural à habitat dispersé,330
34802,34802,95678,Villiers-Adam,Île-de-France,Val-d'Oise,95840.0,1101.0,95127.0,852,10,86.0,49.064,2.235,4,Ceintures urbaines,330
34803,34803,95680,Villiers-le-Bel,Île-de-France,Val-d'Oise,95400.0,1112.0,93005.0,28836,7,3956.0,49.008,2.388,1,Grands centres urbains,326
34804,34804,95682,Villiers-le-Sec,Île-de-France,Val-d'Oise,95720.0,1112.0,93005.0,196,3,61.0,49.072,2.390,6,Rural à habitat dispersé,326


Octave : code_insee =>insee_code;
    code_cluster => indicateur du cluster (de 1 à 330)

